### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from accounts import Account

In [4]:
account = Account.get("Sam")
account

Account(name='sam', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [6]:
account.buy_shares("AAPL", 10, "Because I want to invest in Apple")

'Completed. Latest details:\n{"name": "sam", "balance": 9619.24, "strategy": "", "holdings": {"AAPL": 10}, "transactions": [{"symbol": "AAPL", "quantity": 10, "price": 38.076, "timestamp": "2026-03-27 21:06:17", "rationale": "Because I want to invest in Apple"}], "portfolio_value_time_series": [["2026-03-27 21:06:17", 9839.24]], "total_portfolio_value": 9839.24, "total_profit_loss": -160.76000000000022}'

In [7]:
account.report()

'{"name": "sam", "balance": 9619.24, "strategy": "", "holdings": {"AAPL": 10}, "transactions": [{"symbol": "AAPL", "quantity": 10, "price": 38.076, "timestamp": "2026-03-27 21:06:17", "rationale": "Because I want to invest in Apple"}], "portfolio_value_time_series": [["2026-03-27 21:06:17", 9839.24], ["2026-03-27 21:06:39", 10379.24]], "total_portfolio_value": 10379.24, "total_profit_loss": 379.2399999999998}'

In [8]:
account.list_transactions()

[{'symbol': 'AAPL',
  'quantity': 10,
  'price': 38.076,
  'timestamp': '2026-03-27 21:06:17',
  'rationale': 'Because I want to invest in Apple'}]

### Now we write an MCP server and use it directly!

In [15]:
params = {
    "command": "uv",
    "args": [
        "run",
        "accounts_server.py"
    ]
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    mcp_tools = await mcp_server.list_tools()

In [16]:
print(mcp_tools)

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n    \n    Args:\n        name(str): The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None), Tool(name='get_holdings', description='Get the holdings of the given account user.\n    Args:\n        name(str): The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None), Tool(name='buy_shares', description='Buy shares of a stock for the given account user.\n    Args:\n        name(str): The name of the account\n        symbol(str): The stock symbol to buy\n        quantity(int): The number of shares to buy\n        rationale(str): The rationale for buying the shares\n    ', inputSchema={'properties': {'name': {'ti

In [17]:
instructions = "You are able to manage an account for a client, and answer questions about the account. You can also buy shares for the client, and list transactions. Use the tools at your disposal to manage the account and answer questions about it."
request = "My name is Jeff, and my account is managed by Sam, and it's in his name. Can you please tell me what is the balance in my account and what are my holdings in that account?"
model = 'gpt-4o-mini'

In [19]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="Account Manager", model=model, instructions=instructions, mcp_servers=[mcp_server])
    with trace("account_manager"):
        response = await Runner.run(agent, request)
    display(Markdown(response.final_output))


Your account balance is **$9,619.24**. You currently hold **10 shares of AAPL (Apple Inc.)** in your account.

### Now let's build our own MCP Client

In [20]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n    \n    Args:\n        name(str): The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None), Tool(name='get_holdings', description='Get the holdings of the given account user.\n    Args:\n        name(str): The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None), Tool(name='buy_shares', description='Buy shares of a stock for the given account user.\n    Args:\n        name(str): The name of the account\n        symbol(str): The stock symbol to buy\n        quantity(int): The number of shares to buy\n        rationale(str): The rationale for buying the shares\n    ', inputSchema={'properties': {'name': {'ti

In [21]:
request = "My name is Sam and my account is under the name Sam. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(
        name="account_manager",
        instructions=instructions,
        model=model,
        tools=openai_tools
    )
    response = await Runner.run(agent, request)
    display(Markdown(response.final_output))

Your current balance is $9,619.24.

In [22]:
context = await read_accounts_resource("Sam")
print(context)

{"name": "sam", "balance": 9619.24, "strategy": "", "holdings": {"AAPL": 10}, "transactions": [{"symbol": "AAPL", "quantity": 10, "price": 38.076, "timestamp": "2026-03-27 21:06:17", "rationale": "Because I want to invest in Apple"}], "portfolio_value_time_series": [["2026-03-27 21:06:17", 9839.24], ["2026-03-27 21:06:39", 10379.24], ["2026-03-27 22:13:26", 10339.24]], "total_portfolio_value": 10339.24, "total_profit_loss": 339.2399999999998}


In [23]:
from accounts import Account
Account.get("Sam").report()

'{"name": "sam", "balance": 9619.24, "strategy": "", "holdings": {"AAPL": 10}, "transactions": [{"symbol": "AAPL", "quantity": 10, "price": 38.076, "timestamp": "2026-03-27 21:06:17", "rationale": "Because I want to invest in Apple"}], "portfolio_value_time_series": [["2026-03-27 21:06:17", 9839.24], ["2026-03-27 21:06:39", 10379.24], ["2026-03-27 22:13:26", 10339.24], ["2026-03-27 22:16:25", 10029.24]], "total_portfolio_value": 10029.24, "total_profit_loss": 29.23999999999978}'

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>